# 📊 Notebook 01 — Khám phá Dataset ShanghaiTech Part A

> **Mục tiêu**: Hiểu rõ cấu trúc và đặc điểm của dataset ShanghaiTech Part A trước khi đưa vào huấn luyện.

---

## Nội dung
1. Tải và kiểm tra cấu trúc thư mục
2. Thống kê phân bố số người (histogram)
3. Hiển thị ảnh mẫu + Ground Truth density map
4. Overlay heatmap mật độ lên ảnh gốc
5. So sánh ảnh đám đông thưa vs dày đặc

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import scipy.io as sio
from PIL import Image
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Thêm thư mục gốc project vào Python path
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

print(f'📁 Project root: {PROJECT_ROOT}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CẤU HÌNH ĐƯỜNG DẪN
# ─────────────────────────────────────────────────────────────
import os

# Thay đường dẫn này cho phù hợp với máy của bạn
SHANGHAITECH_ROOT = os.path.expanduser(
    '~/.cache/kagglehub/datasets/tthien/shanghaitech/versions/1/ShanghaiTech'
)
PART_A_ROOT = os.path.join(SHANGHAITECH_ROOT, 'part_A')

TRAIN_IMG_DIR   = os.path.join(PART_A_ROOT, 'train_data', 'images')
TRAIN_GT_DIR    = os.path.join(PART_A_ROOT, 'train_data', 'ground_truth')
TEST_IMG_DIR    = os.path.join(PART_A_ROOT, 'test_data', 'images')
TEST_GT_DIR     = os.path.join(PART_A_ROOT, 'test_data', 'ground_truth')

# Cũng có thể dùng thư mục dữ liệu đã tiền xử lý
PROCESSED_DIR   = os.path.join(PROJECT_ROOT, 'DuLieuDaXuLy', 'part_A', 'part_1')

print('📂 Cấu trúc thư mục:')
for d in [TRAIN_IMG_DIR, TRAIN_GT_DIR, TEST_IMG_DIR, TEST_GT_DIR]:
    exists = os.path.isdir(d)
    count  = len(os.listdir(d)) if exists else 0
    status = f'✅ {count} files' if exists else '❌ Không tìm thấy'
    print(f'  {os.path.relpath(d, SHANGHAITECH_ROOT):50s}  {status}')

## 1. Thống kê Số Người trong Dataset

In [ ]:
def load_gt_count(gt_mat_path):
    """Đọc số lượng người từ file Ground Truth .mat của ShanghaiTech."""
    mat = sio.loadmat(gt_mat_path, squeeze_me=True, struct_as_record=False)
    ann = mat['image_info'].location  # tọa độ từng người
    return len(ann)

train_imgs = sorted(os.listdir(TRAIN_IMG_DIR))
test_imgs  = sorted(os.listdir(TEST_IMG_DIR))

train_counts, test_counts = [], []

for fname in train_imgs:
    stem = os.path.splitext(fname)[0]          # IMG_xxx
    mat  = os.path.join(TRAIN_GT_DIR, f'GT_{stem}.mat')
    if os.path.exists(mat):
        train_counts.append(load_gt_count(mat))

for fname in test_imgs:
    stem = os.path.splitext(fname)[0]
    mat  = os.path.join(TEST_GT_DIR, f'GT_{stem}.mat')
    if os.path.exists(mat):
        test_counts.append(load_gt_count(mat))

print(f'Tập Train: {len(train_counts)} ảnh')
print(f'  Trung bình: {np.mean(train_counts):.0f} người/ảnh')
print(f'  Min: {min(train_counts)} — Max: {max(train_counts)}')
print()
print(f'Tập Test : {len(test_counts)} ảnh')
print(f'  Trung bình: {np.mean(test_counts):.0f} người/ảnh')
print(f'  Min: {min(test_counts)} — Max: {max(test_counts)}')

In [ ]:
# ─── Histogram phân bố số người ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, counts, label, color in zip(
    axes,
    [train_counts, test_counts],
    ['Train Set', 'Test Set'],
    ['#5bc8f5', '#f5a623']
):
    ax.set_facecolor('#1a1a2e')
    ax.hist(counts, bins=20, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(np.mean(counts), color='red', linestyle='--', linewidth=1.5,
               label=f'Trung bình: {np.mean(counts):.0f}')
    ax.set_title(f'ShanghaiTech Part A — {label}', color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Số người trong ảnh', color='white')
    ax.set_ylabel('Số lượng ảnh', color='white')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#2a2a4e', labelcolor='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

plt.tight_layout()
plt.savefig('histogram_crowd_count.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('📊 Histogram đã lưu: histogram_crowd_count.png')

## 2. Hiển thị Ảnh Mẫu + Ground Truth Density Map

In [ ]:
def make_density_from_mat(gt_mat_path, img_shape, sigma=15):
    """Tạo density map từ tọa độ điểm người trong file .mat."""
    mat  = sio.loadmat(gt_mat_path, squeeze_me=True, struct_as_record=False)
    locs = mat['image_info'].location   # (N, 2) — [x, y]
    density = np.zeros(img_shape[:2], dtype=np.float32)
    if locs.ndim == 1:  # chỉ 1 người
        locs = locs[np.newaxis, :]
    for loc in locs:
        x, y = int(loc[0]), int(loc[1])
        if 0 <= y < img_shape[0] and 0 <= x < img_shape[1]:
            density[y, x] = 1.0
    density = gaussian_filter(density, sigma=sigma)
    return density


def show_sample_with_heatmap(img_path, gt_mat_path, ax_row, sigma=15, alpha=0.6):
    img     = np.asarray(Image.open(img_path).convert('RGB'))
    density = make_density_from_mat(gt_mat_path, img.shape, sigma=sigma)
    count   = int(density.sum())

    dmax = density.max()
    if dmax > 0:
        density_norm = density / dmax
    else:
        density_norm = density

    # Overlay heatmap
    colormap = plt.cm.jet(density_norm)[:, :, :3]
    colormap = (colormap * 255).astype(np.uint8)
    overlay  = (alpha * colormap + (1 - alpha) * img).astype(np.uint8)

    ax_row[0].imshow(img)
    ax_row[0].set_title(f'Ảnh gốc  •  {count} người', color='white', fontsize=10)
    ax_row[0].axis('off')

    ax_row[1].imshow(density, cmap='jet')
    ax_row[1].set_title('Density map (Ground Truth)', color='white', fontsize=10)
    ax_row[1].axis('off')

    ax_row[2].imshow(overlay)
    ax_row[2].set_title('Heatmap Overlay', color='white', fontsize=10)
    ax_row[2].axis('off')


# ─── Chọn 3 ảnh để hiển thị ─────────────────────────────────────────
sample_fnames = train_imgs[:3]
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('ShanghaiTech Part A — Ground Truth Density Maps', color='white',
             fontsize=15, fontweight='bold', y=0.98)

for i, fname in enumerate(sample_fnames):
    stem   = os.path.splitext(fname)[0]
    img_p  = os.path.join(TRAIN_IMG_DIR, fname)
    mat_p  = os.path.join(TRAIN_GT_DIR, f'GT_{stem}.mat')
    for ax in axes[i]:
        ax.set_facecolor('#1a1a2e')
    show_sample_with_heatmap(img_p, mat_p, axes[i])

plt.tight_layout()
plt.savefig('sample_density_maps.png', dpi=100, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('🖼️  Ảnh mẫu đã lưu: sample_density_maps.png')

## 3. So sánh Đám đông Thưa vs Dày đặc

In [ ]:
# Tìm ảnh thưa nhất và dày đặc nhất trong tập train
pairs = list(zip(train_counts, train_imgs))
pairs.sort(key=lambda x: x[0])

sparse_count, sparse_fname   = pairs[0]
dense_count,  dense_fname    = pairs[-1]
median_count, median_fname   = pairs[len(pairs)//2]

print(f'Thưa nhất  : {sparse_fname}  →  {sparse_count} người')
print(f'Trung bình : {median_fname}  →  {median_count} người')
print(f'Dày đặc nhất: {dense_fname}  →  {dense_count} người')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('So sánh: Đám đông Thưa → Trung bình → Dày đặc', color='white',
             fontsize=14, fontweight='bold')

for (col, fname, lbl) in [(0, sparse_fname, f'Thưa ({sparse_count} người)'),
                          (1, median_fname, f'Trung bình ({median_count} người)'),
                          (2, dense_fname,  f'Dày đặc ({dense_count} người)')]:
    stem  = os.path.splitext(fname)[0]
    img_p = os.path.join(TRAIN_IMG_DIR, fname)
    mat_p = os.path.join(TRAIN_GT_DIR,  f'GT_{stem}.mat')

    img     = np.asarray(Image.open(img_p).convert('RGB'))
    density = make_density_from_mat(mat_p, img.shape)

    for ax in [axes[0][col], axes[1][col]]:
        ax.set_facecolor('#1a1a2e')

    axes[0][col].imshow(img)
    axes[0][col].set_title(lbl, color='white', fontsize=11, fontweight='bold')
    axes[0][col].axis('off')

    axes[1][col].imshow(density, cmap='hot')
    axes[1][col].set_title('Density Map', color='white', fontsize=10)
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('sparse_vs_dense_crowd.png', dpi=100, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('🖼️  So sánh đã lưu: sparse_vs_dense_crowd.png')

## 4. Kiểm tra Dữ liệu Đã Tiền xử lý (DuLieuDaXuLy)

In [ ]:
# Kiểm tra thư mục dữ liệu đã tiền xử lý (sau khi chạy preprocess_shtech.py)
train_processed = os.path.join(PROCESSED_DIR, 'train')
test_processed  = os.path.join(PROCESSED_DIR, 'test')

if os.path.isdir(train_processed):
    processed_imgs = [f for f in os.listdir(train_processed) if f.endswith('.jpg')]
    print(f'✅ Dữ liệu train đã xử lý: {len(processed_imgs)} crops JPG')
    
    # Xem thử 1 crop đã xử lý
    sample_crop = processed_imgs[0]
    img   = Image.open(os.path.join(train_processed, sample_crop))
    img_arr = np.asarray(img)
    print(f'   Kích thước crop mẫu: {img_arr.shape}')
    print(f'   Tên file: {sample_crop}')
    
    # Đọc density CSV tương ứng
    csv_path = os.path.join(train_processed + '_den', sample_crop.replace('.jpg', '.csv'))
    if os.path.exists(csv_path):
        den = pd.read_csv(csv_path, header=None).values
        print(f'   Density CSV shape: {den.shape}  |  Sum: {den.sum():.2f}')
else:
    print('⚠️  Chưa có dữ liệu tiền xử lý. Hãy chạy: bash run_pipeline.sh --steps preprocess')

---
> **Tiếp theo**: Chạy `bash run_pipeline.sh --steps train` để huấn luyện mô hình,
> sau đó mở `03_evaluate_results.ipynb` để phân tích kết quả.